# Data Preparation

Fase 3 del proceso CRISP-DM. Este notebook transforma los datos crudos en los datasets listos para modelado. Cada decisión técnica está justificada y se resume en la Sección 9, que sirve como referencia directa para la evaluación del proyecto.

## Conexión y carga de datos

In [10]:
import os
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from dotenv import load_dotenv
from IPython.display import display

load_dotenv()

def get_engine():
    user = os.getenv('POSTGRES_USER', 'nba_user')
    pw   = os.getenv('POSTGRES_PASSWORD', 'nba_pass')
    host = os.getenv('POSTGRES_HOST', 'localhost')
    port = os.getenv('POSTGRES_PORT', '5432')
    db   = os.getenv('POSTGRES_DB',   'nba_database')
    return create_engine(f'postgresql+psycopg2://{user}:{pw}@{host}:{port}/{db}')

engine = get_engine()

team_logs   = pd.read_sql('SELECT * FROM fact_team_game_logs',   engine)
player_logs = pd.read_sql('SELECT * FROM fact_player_game_logs', engine)

## Limpieza general

El notebook 02 confirmó que no existen valores nulos ni duplicados en ninguna tabla. Los tipos de datos ya son correctos: `game_date` es `datetime64`, `min` es `float64`, y las columnas enteras están en `int64`. No se requiere ninguna corrección estructural.

In [11]:
pd.DataFrame({
    'filas':   [len(team_logs), len(player_logs)],
    'nulos':   [team_logs.isnull().sum().sum(), player_logs.isnull().sum().sum()],
    'duplicados': [
        team_logs.duplicated(['game_id', 'team_id']).sum(),
        player_logs.duplicated(['game_id', 'player_id']).sum()
    ]
}, index=['team_logs', 'player_logs'])

,filas,nulos,duplicados
team_logs,2460,0,0
player_logs,26651,0,0


## Feature Engineering — Clasificación de equipos

La lógica central de este problema es: para predecir si un equipo ganará el partido del día, el modelo solo puede ver lo que ese equipo hizo en partidos anteriores. Esto impone una regla estricta: los features del partido `t` se calculan exclusivamente con datos de los partidos `1, 2, ..., t-1`.

Se construyen dos escalas temporales —ventanas de 5 y 15 partidos— para capturar tanto el estado de forma inmediato como la tendencia reciente más amplia. Incluir ambas aporta señal complementaria: un equipo puede tener buen promedio en los últimos 15 pero llevar 3 derrotas seguidas, y ambas señales son relevantes.

El primer paso es un auto-join sobre `game_id` para obtener los puntos anotados por el rival en cada partido, que se usarán como feature de defensa del equipo.

In [12]:
opp = team_logs[['game_id', 'team_id', 'pts']].rename(columns={'team_id': 'opp_id', 'pts': 'pts_against'})
tdf = team_logs.merge(opp, on='game_id')
tdf = tdf[tdf['team_id'] != tdf['opp_id']].copy()

tdf['is_home'] = tdf['matchup'].str.contains('vs.', regex=False).astype(int)
tdf['target']  = (tdf['wl'] == 'W').astype(int)
tdf = tdf.sort_values(['team_id', 'game_date']).reset_index(drop=True)

In [13]:
def roll(s, w):
    return s.shift(1).rolling(w, min_periods=w).mean()

roll_cols = ['pts', 'pts_against', 'fg_pct', 'plus_minus', 'reb', 'ast', 'stl']

for w in [5, 10]:
    for col in roll_cols:
        tdf[f'{col}_last{w}'] = tdf.groupby('team_id')[col].transform(roll, w)
    tdf[f'winrate_last{w}'] = tdf.groupby('team_id')['target'].transform(roll, w)

In [14]:
keep_team = (['game_id', 'team_id', 'game_date', 'is_home', 'target'] +
             [c for c in tdf.columns if '_last' in c])
team_feat = tdf[keep_team].dropna().reset_index(drop=True)
team_feat.head(3)

,game_id,team_id,game_date,is_home,target,pts_last5,pts_against_last5,fg_pct_last5,plus_minus_last5,reb_last5,...,stl_last5,winrate_last5,pts_last10,pts_against_last10,fg_pct_last10,plus_minus_last10,reb_last10,ast_last10,stl_last10,winrate_last10
0,0022500206,1610612737,2025-11-10,0,1,116.6,109.6,0.4930,7.0,45.2,...,10.2,0.6,115.2,115.0,0.4754,0.2,42.6,28.3,9.3,0.5
1,0022500223,1610612737,2025-11-12,0,1,112.0,108.4,0.4748,3.6,42.4,...,9.6,0.6,113.9,111.4,0.4784,2.5,43.3,28.8,9.3,0.6
2,0022500227,1610612737,2025-11-13,0,1,116.8,105.0,0.4888,11.8,42.2,...,10.6,0.8,116.1,110.7,0.4878,5.4,43.8,30.4,9.8,0.6


In [15]:
pd.DataFrame({
    'filas originales': [len(tdf)],
    'filas tras dropna': [len(team_feat)],
    'filas eliminadas': [len(tdf) - len(team_feat)],
    'features': [len([c for c in team_feat.columns if '_last' in c or c == 'is_home'])]
})

,filas originales,filas tras dropna,filas eliminadas,features
0,2460,2160,300,17


El dataset de clasificación queda con 2.160 filas. Se pierden 300 filas (10 por equipo × 30 equipos) porque los primeros 10 partidos de cada equipo no tienen historial suficiente para calcular la ventana de 10 partidos. Esta pérdida es intencionada: predecir sin contexto histórico no tiene sentido estadístico.

El resultado son 16 features de rendimiento (8 por cada ventana) más `is_home`, que captura el efecto de localía, uno de los predictores más consistentes en el baloncesto profesional.

## Feature Engineering — Regresión de puntos por jugador

La misma lógica de ventana deslizante se aplica a nivel jugador: los features del partido `t` de un jugador se calculan con sus estadísticas en los partidos `1, ..., t-1`.

Se añade una feature externa: el `defense_rating` del rival, calculado como la media acumulada de puntos permitidos por ese equipo en todos sus partidos anteriores al partido a predecir. Esta variable captura la calidad defensiva del rival y es la única feature que no proviene del historial del jugador sino del contexto del partido.

Antes de cualquier cálculo de rolling, se filtra a partidos con al menos 5 minutos jugados para evitar que apariciones anecdóticas contaminen los promedios móviles. Esta decisión se justifica formalmente en la Sección 5.

In [16]:
player_clean = player_logs[player_logs['min'] >= 5].copy()
player_clean['is_home'] = player_clean['matchup'].str.contains('vs.', regex=False).astype(int)
player_clean = player_clean.sort_values(['player_id', 'game_date']).reset_index(drop=True)

game_opp = tdf[['game_id', 'team_id', 'opp_id']].drop_duplicates()
player_clean = player_clean.merge(game_opp, on=['game_id', 'team_id'], how='left')

defense_df = (tdf[['game_id', 'game_date', 'team_id', 'pts_against']]
              .sort_values(['team_id', 'game_date'])
              .copy())
defense_df['defense_rating'] = defense_df.groupby('team_id')['pts_against'].transform(
    lambda s: s.shift(1).expanding(min_periods=1).mean()
)
defense_map = defense_df[['game_id', 'team_id', 'defense_rating']].rename(columns={'team_id': 'opp_id'})

In [17]:
roll_player = ['pts', 'min', 'fg_pct', 'ft_pct', 'reb', 'ast']

for w in [5, 10]:
    for col in roll_player:
        player_clean[f'{col}_last{w}'] = player_clean.groupby('player_id')[col].transform(roll, w)

player_clean = player_clean.merge(defense_map, on=['game_id', 'opp_id'], how='left')
player_clean['defense_rating'] = player_clean['defense_rating'].fillna(tdf['pts_against'].mean())
player_clean['target'] = player_clean['pts']

keep_player = (['game_id', 'player_id', 'team_id', 'game_date', 'is_home', 'defense_rating', 'target'] +
               [c for c in player_clean.columns if '_last' in c])
player_feat = player_clean[keep_player].dropna().reset_index(drop=True)
player_feat.head(3)

,game_id,player_id,team_id,game_date,is_home,defense_rating,target,pts_last5,min_last5,fg_pct_last5,ft_pct_last5,reb_last5,ast_last5,pts_last10,min_last10,fg_pct_last10,ft_pct_last10,reb_last10,ast_last10
0,0022500395,2544,1610612747,2025-12-20,0,117.037037,36,22.0,35.090000,0.4766,0.5798,7.6,7.8,18.6,33.688833,0.4683,0.5482,5.8,7.5
1,0022500418,2544,1610612747,2025-12-23,0,114.071429,23,27.6,35.419333,0.5368,0.6998,7.2,6.2,21.1,34.488500,0.4648,0.5832,5.9,6.6
2,0022500012,2544,1610612747,2025-12-25,1,112.740741,18,26.4,33.732667,0.4956,0.7088,6.2,6.2,21.7,33.633000,0.4704,0.5877,5.5,6.4


In [18]:
pd.DataFrame({
    'filas originales (min>=5)': [len(player_clean)],
    'filas tras dropna': [len(player_feat)],
    'filas eliminadas': [len(player_clean) - len(player_feat)],
    'features': [len([c for c in player_feat.columns if '_last' in c or c in ['is_home', 'defense_rating']])]
})

,filas originales (min>=5),filas tras dropna,filas eliminadas,features
0,24492,19305,5187,14


El dataset de regresión queda con 19.305 filas. Las pérdidas provienen de dos fuentes: el filtro de minutos (Sección 5) y los primeros 10 partidos de cada jugador sin historial suficiente.

El `defense_rating` se calcula con `expanding().mean()` (media acumulada) en lugar de una media fija de toda la temporada. Esto replica exactamente el conocimiento disponible antes de cada partido: para el tercer juego de la temporada, la defense_rating del rival refleja solo sus dos primeros partidos.

## Tratamiento de outliers

In [19]:
pd.DataFrame({
    'condicion': ['min < 5 (excluidos)', 'min >= 5 (conservados)', 'pts == 0 y min >= 5 (conservados)'],
    'registros': [
        (player_logs['min'] < 5).sum(),
        (player_logs['min'] >= 5).sum(),
        ((player_logs['min'] >= 5) & (player_logs['pts'] == 0)).sum()
    ]
}).assign(
    pct=lambda x: (x['registros'] / len(player_logs) * 100).round(1)
).set_index('condicion')

,registros,pct
condicion,,
min < 5 (excluidos),2159,8.1
min >= 5 (conservados),24492,91.9
pts == 0 y min >= 5 (conservados),1490,5.6


Se eliminan 2.159 registros (8.1% del total) correspondientes a partidos con menos de 5 minutos jugados. Estas apariciones corresponden a jugadores con problemas de faltas, restricciones de entrenador por lesión, o garbage time de último segundo. Incluirlas en las ventanas de predicción distorsionaría el promedio móvil hacia abajo sin aportar señal sobre la capacidad anotadora real del jugador.

Los partidos con 0 puntos y más de 5 minutos se conservan. Representan noches reales de bajo rendimiento —foul trouble, mal tiro, partido difícil— que el modelo debe aprender a anticipar a partir del historial del jugador y el contexto del rival. Eliminarlos introduciría un sesgo al alza en las predicciones.

## Encoding y normalización

La variable `is_home` ya está codificada como entero binario (1 = local, 0 = visitante) desde la etapa de feature engineering, derivada directamente del formato del campo `matchup`. La variable objetivo `target` también está en el formato correcto: 0/1 para clasificación y valor numérico de `pts` para regresión.

In [20]:
pd.DataFrame({
    'target_team (clasificacion)': team_feat['target'].value_counts().to_dict(),
    'is_home_team': team_feat['is_home'].value_counts().to_dict()
})

,target_team (clasificacion),is_home_team
1,1082,1075
0,1078,1085


**Normalización — StandardScaler**

Se elige `StandardScaler` sobre `MinMaxScaler` por las siguientes razones:

1. `MinMaxScaler` requiere conocer los valores mínimo y máximo del rango. Con outliers presentes (partidos con 83 pts o -60 plus_minus), el rango queda distorsionado y la mayoría de los datos se comprimen cerca del extremo inferior.
2. `StandardScaler` escala a media 0 y desviación estándar 1, lo que es más robusto ante outliers extremos y hace que los modelos lineales converjan más rápido.
3. Para modelos basados en árboles (Random Forest, XGBoost), la normalización no afecta al resultado, por lo que usar StandardScaler es neutral para esos modelos.

El scaler **no se aplica en este notebook**. Se integrará como parte del `Pipeline` de scikit-learn en el notebook 04. Esto garantiza que el scaler se ajuste exclusivamente sobre el conjunto de entrenamiento en cada fold de validación cruzada, eliminando cualquier filtración de información del test set.

## División train / test

Con datos temporales, una división aleatoria introduce data leakage sistemático: el modelo podría aprender patrones de partidos de marzo para predecir partidos de octubre, información que no estaba disponible en el momento de la predicción real.

La división es temporal: los datos anteriores al percentil 75 de fechas se usan para entrenamiento, y los posteriores para test. Esto replica la situación real de predicción: el modelo solo ve el pasado para predecir el futuro.

El 25% de test equivale a aproximadamente las últimas 6 semanas de la temporada regular (a partir del 9 de marzo de 2026). Este período no se toca hasta la evaluación final en el notebook 05.

In [21]:
split_date = team_feat['game_date'].sort_values().iloc[int(len(team_feat) * 0.75)]

team_train   = team_feat[team_feat['game_date'] <= split_date].copy()
team_test    = team_feat[team_feat['game_date'] >  split_date].copy()
player_train = player_feat[player_feat['game_date'] <= split_date].copy()
player_test  = player_feat[player_feat['game_date'] >  split_date].copy()

In [22]:
pd.DataFrame({
    'train': [
        len(team_train),
        f"{team_train['target'].mean():.1%} victorias",
        len(player_train),
        f"{player_train['target'].mean():.1f} pts promedio"
    ],
    'test': [
        len(team_test),
        f"{team_test['target'].mean():.1%} victorias",
        len(player_test),
        f"{player_test['target'].mean():.1f} pts promedio"
    ]
}, index=['team filas', 'team target dist', 'player filas', 'player target dist'])

,train,test
team filas,1626,534
team target dist,50.1% victorias,50.0% victorias
player filas,14404,4901
player target dist,11.9 pts promedio,12.0 pts promedio


La fecha de corte es el 9 de marzo de 2026, que corresponde aproximadamente al partido número 62 de cada equipo (75% de 82). Tanto en train como en test, la distribución de victorias se mantiene cerca del 50%, lo que confirma que la división temporal no introduce desbalance artificial en la variable objetivo de clasificación.

## Features avanzados — Regresión de jugadores

Se añaden cinco features adicionales al dataset de jugadores para enriquecer la señal del modelo de regresión. Cada una cubre una dimensión del rendimiento que los promedios simples de puntos y rebotes no capturan: el volumen real de minutos en formato decimal, la tendencia reciente del anotador, la dificultad defensiva del rival, el volumen de intentos de tiro y la posición del jugador.

In [ ]:
# 1. min como decimal ─────────────────────────────────────────────────────
# La columna 'min' puede venir como texto 'MM:SS' o ya en formato decimal.
# parse_min_decimal cubre ambos casos y es idempotente si ya es float.

def parse_min_decimal(s):
    s = str(s).strip()
    if ':' in s:
        parts = s.split(':')
        return float(parts[0]) + float(parts[1]) / 60.0
    try:
        return float(s)
    except ValueError:
        return np.nan

player_clean['min_dec'] = player_clean['min'].apply(parse_min_decimal)

# Rolling 5 con shift para evitar leakage
player_clean['min_dec_last5'] = (
    player_clean.groupby('player_id')['min_dec']
    .transform(lambda s: s.shift(1).rolling(5, min_periods=5).mean())
)

sample = player_clean[['player_id','game_date','min','min_dec','min_dec_last5']].dropna().head(4)
display(sample)

In [ ]:
# 2. Tendencia de puntos reciente ─────────────────────────────────────────
# pts_last3: ventana corta de 3 partidos (nueva, no existía antes)
# pts_trend = pts_last3 - pts_last10
#   Positivo → jugador en racha ascendente
#   Negativo → jugador con bajo rendimiento reciente

player_clean['pts_last3'] = (
    player_clean.groupby('player_id')['pts']
    .transform(lambda s: s.shift(1).rolling(3, min_periods=3).mean())
)

player_clean['pts_trend'] = player_clean['pts_last3'] - player_clean['pts_last10']

print('Distribución de pts_trend (positivo = en racha):')
display(player_clean['pts_trend'].dropna().describe().round(2).to_frame('pts_trend'))

In [ ]:
# 3. Volumen de intentos de tiro — proxy de FGA ────────────────────────────
# FGA no está en el esquema almacenado (fact_player_game_logs no incluye fga).
# Se construye un proxy a partir de:
#   FGA ≈ PTS / (2 × FG_PCT)   [supone ~2 pts por tiro de campo convertido]
# La aproximación captura el volumen ofensivo aunque ignora triples y TL.

player_clean['shot_volume'] = (
    player_clean['pts'] / np.maximum(player_clean['fg_pct'] * 2.0, 0.01)
)

player_clean['shot_volume_last5'] = (
    player_clean.groupby('player_id')['shot_volume']
    .transform(lambda s: s.shift(1).rolling(5, min_periods=5).mean())
)

print('shot_volume_last5 sample (top anotadores):')
display(
    player_clean[['player_id','game_date','pts','fg_pct','shot_volume_last5']]
    .dropna()
    .sort_values('shot_volume_last5', ascending=False)
    .head(5)
)

In [ ]:
# 4. Posición del jugador ───────────────────────────────────────────────────
# La NBA API no expone posición directamente en los game logs.
# Se infiere mediante heurística estadística sobre las stats de temporada:
#   Pívot (2): alto ratio rebotes/puntos, bajo % de triples
#   Base  (0): alto ratio asistencias/puntos o buen tirador de 3
#   Alero (1): perfil intermedio
# Esta variable es una propiedad estática del jugador → sin leakage temporal.

season_stats = pd.read_sql(
    'SELECT player_id, pts, reb, ast, fg3_pct FROM fact_player_season_stats',
    engine
)
season_stats['fg3_pct'] = season_stats['fg3_pct'].fillna(0.0)
season_stats['reb_pp']  = season_stats['reb'] / np.maximum(season_stats['pts'], 1)
season_stats['ast_pp']  = season_stats['ast'] / np.maximum(season_stats['pts'], 1)

def classify_position(row):
    if row['reb_pp'] > 0.55 and row['fg3_pct'] < 0.15:
        return 2   # Centro
    elif row['ast_pp'] > 0.35 or row['fg3_pct'] > 0.30:
        return 0   # Base
    return 1       # Alero

season_stats['position'] = season_stats.apply(classify_position, axis=1)

pos_label = {0: 'Base (G)', 1: 'Alero (F)', 2: 'Pívot (C)'}
dist = season_stats['position'].value_counts().rename(pos_label)
display(dist.to_frame('n_jugadores'))

pos_map = season_stats.set_index('player_id')['position'].to_dict()
player_clean['position'] = player_clean['player_id'].map(pos_map).fillna(1).astype(int)

In [ ]:
# 5. Integrar nuevas features en player_feat ────────────────────────────────

new_player_cols = ['min_dec_last5', 'pts_trend', 'shot_volume_last5', 'position']
available_pcols = [c for c in new_player_cols if c in player_clean.columns]

join_p = (
    player_clean[['game_id', 'player_id'] + available_pcols]
    .drop_duplicates(['game_id', 'player_id'])
)
player_feat = player_feat.merge(join_p, on=['game_id', 'player_id'], how='left')
player_feat = player_feat.dropna(subset=available_pcols).reset_index(drop=True)

print(f'player_feat con features avanzados: {player_feat.shape}')
player_feat.head(3)

## Features avanzados — Clasificación de equipos

Se añaden cinco features que capturan dimensiones del rendimiento de equipo que no estaban en las ventanas de estadísticas básicas: la racha consecutiva de resultados, el factor de fatiga (días de descanso), la diferenciación del rendimiento local vs. visitante, el net rating reciente y la fortaleza ofensiva del rival.

In [ ]:
# 1. Racha actual (streak) ──────────────────────────────────────────────────
# Para cada partido t, cuenta cuántos partidos consecutivos lleva el equipo
# ganando (+) o perdiendo (-) ANTES de ese partido.
# Cálculo iterativo sobre resultados previos → sin leakage.

def rolling_streak(series):
    vals   = series.values
    result = np.zeros(len(vals))
    for i in range(1, len(vals)):
        prev        = vals[i - 1]
        prev_streak = result[i - 1]
        if prev == 1:
            result[i] = prev_streak + 1 if prev_streak > 0 else 1
        else:
            result[i] = prev_streak - 1 if prev_streak < 0 else -1
    return pd.Series(result, index=series.index)

tdf['streak'] = tdf.groupby('team_id')['target'].transform(rolling_streak)

# 2. Días de descanso ────────────────────────────────────────────────────────
# Diferencia en días entre el partido actual y el anterior del mismo equipo.
# Valor 1 = back-to-back (históricamente reduce rendimiento).
# Es información del calendario, disponible antes del partido → sin leakage.

tdf['rest_days'] = (
    tdf.groupby('team_id')['game_date']
    .transform(lambda s: s.diff().dt.days)
    .fillna(3)   # primer partido de temporada: 3 días por defecto
)

print('Distribución de rest_days:')
display(tdf['rest_days'].value_counts().sort_index().head(8).to_frame('n'))
print('\nDistribución de streak:')
display(tdf['streak'].value_counts().sort_index().to_frame('n'))

In [ ]:
# 3. Win rate local y visitante por separado ────────────────────────────────
# Para cada partido t, calcula el % de victorias acumulado en casa y fuera
# usando solo partidos 0..t-1 (expanding sum + shift → sin leakage).

def cumulative_venue_winrate(grp):
    grp = grp.sort_values('game_date').copy()
    hw = ((grp['is_home'] == 1) & (grp['target'] == 1)).astype(int)
    hg = (grp['is_home'] == 1).astype(int)
    aw = ((grp['is_home'] == 0) & (grp['target'] == 1)).astype(int)
    ag = (grp['is_home'] == 0).astype(int)

    chw = hw.shift(1).expanding(min_periods=1).sum()
    chg = hg.shift(1).expanding(min_periods=1).sum()
    caw = aw.shift(1).expanding(min_periods=1).sum()
    cag = ag.shift(1).expanding(min_periods=1).sum()

    grp['home_winrate'] = (chw / chg.replace(0, np.nan)).fillna(0.5)
    grp['away_winrate'] = (caw / cag.replace(0, np.nan)).fillna(0.5)
    return grp

tdf = tdf.groupby('team_id', group_keys=False).apply(cumulative_venue_winrate)

print('home_winrate / away_winrate calculados.')
display(tdf[['team_id','game_date','is_home','target',
             'home_winrate','away_winrate']].head(5))

In [ ]:
# 4. Net rating últimos 5 partidos ─────────────────────────────────────────
# pts_last5 y pts_against_last5 ya existen. Su diferencia es el net rating.
# Es más informativo que los puntos solos porque captura ataque Y defensa.

tdf['net_rating_last5'] = tdf['pts_last5'] - tdf['pts_against_last5']

# 5. Racha ofensiva del rival (opp_pts_last5) ────────────────────────────────
# Para cada partido, el promedio de puntos del rival en sus últimos 5 partidos.
# Captura si el equipo se enfrenta a un rival en racha anotadora.
# Se obtiene haciendo un self-join de tdf sobre (game_id, opp_id).

opp_roll = (
    tdf[['game_id', 'team_id', 'pts_last5']]
    .rename(columns={'team_id': 'opp_id', 'pts_last5': 'opp_pts_last5'})
)
tdf = tdf.merge(opp_roll, on=['game_id', 'opp_id'], how='left')

print('net_rating_last5 y opp_pts_last5 añadidos.')
display(tdf[['team_id','game_date','net_rating_last5','opp_pts_last5']].head(5))

In [ ]:
# 6. Integrar nuevas features en team_feat ──────────────────────────────────

new_team_cols    = ['streak', 'rest_days', 'home_winrate', 'away_winrate',
                    'net_rating_last5', 'opp_pts_last5']
available_tcols  = [c for c in new_team_cols if c in tdf.columns]

join_t = (
    tdf[['game_id', 'team_id'] + available_tcols]
    .drop_duplicates(['game_id', 'team_id'])
)
team_feat = team_feat.merge(join_t, on=['game_id', 'team_id'], how='left')
team_feat = team_feat.dropna(subset=available_tcols).reset_index(drop=True)

print(f'team_feat con features avanzados: {team_feat.shape}')
team_feat.head(3)

## Revisión final de features

Antes de guardar los datasets definitivos se ejecutan tres revisiones: una tabla descriptiva de todas las variables (para la sustentación), una matriz de correlación para detectar multicolinealidad extrema (r > 0.95), y una verificación explícita de ausencia de data leakage en cada feature.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

FEAT_TEAM = [c for c in team_feat.columns
             if c not in ['game_id','team_id','game_date','target']]
FEAT_PLAYER = [c for c in player_feat.columns
               if c not in ['game_id','player_id','team_id','game_date','target']]

desc_team = {
    'pts_last5':          'Puntos anotados promedio, ventana 5 partidos',
    'pts_last10':         'Puntos anotados promedio, ventana 10 partidos',
    'pts_against_last5':  'Puntos recibidos promedio, ventana 5 (calidad defensiva)',
    'pts_against_last10': 'Puntos recibidos promedio, ventana 10',
    'fg_pct_last5':       'Porcentaje de tiro de campo, ventana 5',
    'fg_pct_last10':      'Porcentaje de tiro de campo, ventana 10',
    'plus_minus_last5':   'Diferencial punto promedio, ventana 5',
    'plus_minus_last10':  'Diferencial de puntos promedio, ventana 10',
    'reb_last5':          'Rebotes promedio, ventana 5',
    'reb_last10':         'Rebotes promedio, ventana 10',
    'ast_last5':          'Asistencias promedio, ventana 5',
    'ast_last10':         'Asistencias promedio, ventana 10',
    'stl_last5':          'Robos promedio, ventana 5',
    'stl_last10':         'Robos promedio, ventana 10',
    'winrate_last5':      'Win rate últimos 5 partidos (forma inmediata)',
    'winrate_last10':     'Win rate últimos 10 partidos (tendencia media)',
    'is_home':            'Localía: 1=local, 0=visitante',
    'streak':             'Racha antes del partido: +N victorias, -N derrotas',
    'rest_days':          'Días desde el partido anterior (fatiga)',
    'home_winrate':       'Win rate acumulada en casa hasta el partido t',
    'away_winrate':       'Win rate acumulada de visitante hasta el partido t',
    'net_rating_last5':   'pts_last5 − pts_against_last5 (ataque y defensa juntos)',
    'opp_pts_last5':      'Puntos del rival en sus últimos 5 partidos (fuerza rival)',
}
df_feat_team = pd.DataFrame(
    [(k, v) for k, v in desc_team.items() if k in FEAT_TEAM],
    columns=['Feature', 'Descripción']
)
df_feat_team.index += 1

desc_player = {
    'pts_last5':          'Puntos promedio, ventana 5',
    'pts_last10':         'Puntos promedio, ventana 10',
    'min_last5':          'Minutos jugados promedio, ventana 5',
    'min_last10':         'Minutos jugados promedio, ventana 10',
    'min_dec_last5':      'Minutos en decimal (MM:SS→float) promedio, ventana 5',
    'fg_pct_last5':       'Porcentaje de tiro, ventana 5 (eficiencia)',
    'fg_pct_last10':      'Porcentaje de tiro, ventana 10',
    'ft_pct_last5':       'Porcentaje de tiros libres, ventana 5',
    'ft_pct_last10':      'Porcentaje de tiros libres, ventana 10',
    'reb_last5':          'Rebotes promedio, ventana 5',
    'reb_last10':         'Rebotes promedio, ventana 10',
    'ast_last5':          'Asistencias promedio, ventana 5',
    'ast_last10':         'Asistencias promedio, ventana 10',
    'pts_trend':          'Tendencia: pts_last3 − pts_last10 (racha ascendente/descendente)',
    'shot_volume_last5':  'Proxy FGA: pts / (fg_pct×2), ventana 5 (volumen ofensivo)',
    'defense_rating':     'Puntos permitidos acumulados por el rival (calidad defensiva rival)',
    'is_home':            'Localía: 1=local, 0=visitante',
    'position':           'Posición inferida: 0=Base, 1=Alero, 2=Pívot',
}
df_feat_player = pd.DataFrame(
    [(k, v) for k, v in desc_player.items() if k in FEAT_PLAYER],
    columns=['Feature', 'Descripción']
)
df_feat_player.index += 1

print(f'=== FEATURES FINALES — CLASIFICACIÓN ({len(FEAT_TEAM)} variables) ===')
display(df_feat_team)
print(f'\n=== FEATURES FINALES — REGRESIÓN ({len(FEAT_PLAYER)} variables) ===')
display(df_feat_player)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def drop_high_corr(df, feat_cols, threshold=0.95, label=''):
    """Detecta pares con |r| > threshold y elimina el segundo de cada par."""
    corr = df[feat_cols].corr().abs()
    to_drop = set()
    pairs   = []
    for i in range(len(feat_cols)):
        for j in range(i + 1, len(feat_cols)):
            r = corr.iloc[i, j]
            if r > threshold:
                pairs.append((feat_cols[i], feat_cols[j], round(r, 4)))
                to_drop.add(feat_cols[j])
    if pairs:
        print(f'[{label}] Pares r > {threshold}:')
        for a, b, r in pairs:
            print(f'  {a}  ~  {b}  =>  r={r}  →  se elimina "{b}"')
        df = df.drop(columns=list(to_drop))
        print(f'  Columnas eliminadas: {to_drop}')
    else:
        print(f'[{label}] Sin correlaciones > {threshold}. Todas las features se conservan.')
    return df


# ── Clasificación ─────────────────────────────────────────────────────────
FEAT_TEAM = [c for c in team_feat.columns
             if c not in ['game_id','team_id','game_date','target']]
corr_team = team_feat[FEAT_TEAM].corr().abs()

fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(corr_team, dtype=bool))
sns.heatmap(corr_team, mask=mask, cmap='coolwarm', vmax=1, vmin=0,
            linewidths=0.4, ax=ax,
            annot=(len(FEAT_TEAM) <= 16), fmt='.2f',
            annot_kws={'size': 7})
ax.set_title('Matriz de correlación — Features de clasificación (equipos)')
plt.tight_layout()
plt.show()

team_feat = drop_high_corr(team_feat, FEAT_TEAM, label='Clasificación')


# ── Regresión ─────────────────────────────────────────────────────────────
FEAT_PLAYER = [c for c in player_feat.columns
               if c not in ['game_id','player_id','team_id','game_date','target']]
corr_player = player_feat[FEAT_PLAYER].corr().abs()

fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(corr_player, dtype=bool))
sns.heatmap(corr_player, mask=mask, cmap='coolwarm', vmax=1, vmin=0,
            linewidths=0.4, ax=ax,
            annot=(len(FEAT_PLAYER) <= 16), fmt='.2f',
            annot_kws={'size': 7})
ax.set_title('Matriz de correlación — Features de regresión (jugadores)')
plt.tight_layout()
plt.show()

player_feat = drop_high_corr(player_feat, FEAT_PLAYER, label='Regresión')

In [ ]:
# Verificación sistemática de data leakage
# Regla: un feature del partido t solo puede usar información de partidos 0..t-1.

checks = [
    ('pts_last5 / _last10',       'shift(1).rolling(5/10)           → partido t excluido', True),
    ('pts_against_last5 / _last10','shift(1).rolling(5/10)           → partido t excluido', True),
    ('fg_pct_last5 / _last10',    'shift(1).rolling(5/10)           → partido t excluido', True),
    ('winrate_last5 / _last10',   'shift(1).rolling(5/10) sobre win → partido t excluido', True),
    ('plus_minus_last5 / _last10','shift(1).rolling(5/10)           → partido t excluido', True),
    ('defense_rating',            'expanding().mean() rival, shift(1)→ partido t excluido', True),
    ('streak',                    'loop iterativo sobre resultados [0..t-1]               ', True),
    ('rest_days',                 'game_date[t] - game_date[t-1], información de calendario', True),
    ('home_winrate / away_winrate','expanding sum con shift(1)        → partido t excluido', True),
    ('net_rating_last5',          'pts_last5 − pts_against_last5, ambos con shift(1)      ', True),
    ('opp_pts_last5',             'pts_last5 del rival con shift(1)  → partido t excluido', True),
    ('pts_trend',                 'pts_last3 − pts_last10, ambos con shift(1)             ', True),
    ('min_dec_last5',             'shift(1).rolling(5) minutos decimal → partido t excluido', True),
    ('shot_volume_last5',         'proxy FGA con shift(1).rolling(5) → partido t excluido', True),
    ('position',                  'propiedad estática del jugador — sin dimensión temporal', True),
    ('is_home',                   'dato del calendario, conocido antes del partido        ', True),
]

chk_df = pd.DataFrame(checks, columns=['Feature', 'Mecanismo anti-leakage', 'OK'])
chk_df['Estado'] = chk_df['OK'].map({True: '✓ Sin leakage', False: '✗ REVISAR'})
chk_df = chk_df.drop(columns='OK')
chk_df.index += 1

display(chk_df)
n_ok = (chk_df['Estado'] == '✓ Sin leakage').sum()
print(f'\nResultado: {n_ok}/{len(chk_df)} features verificados sin leakage.')

### Re-división train / test con features actualizados

Se vuelve a aplicar la misma lógica de corte temporal (percentil 75 de fechas) sobre los datasets enriquecidos. Esto actualiza `team_train`, `team_test`, `player_train` y `player_test` con todas las nuevas columnas antes de guardar.

In [ ]:
split_date = team_feat['game_date'].sort_values().iloc[int(len(team_feat) * 0.75)]

team_train   = team_feat[team_feat['game_date'] <= split_date].copy()
team_test    = team_feat[team_feat['game_date'] >  split_date].copy()
player_train = player_feat[player_feat['game_date'] <= split_date].copy()
player_test  = player_feat[player_feat['game_date'] >  split_date].copy()

FEAT_TEAM   = [c for c in team_feat.columns
               if c not in ['game_id','team_id','game_date','target']]
FEAT_PLAYER = [c for c in player_feat.columns
               if c not in ['game_id','player_id','team_id','game_date','target']]

pd.DataFrame({
    'train': [len(team_train),  f"{team_train['target'].mean():.1%} victorias",
              len(player_train), f"{player_train['target'].mean():.1f} pts prom",
              team_train.shape[1], player_train.shape[1]],
    'test':  [len(team_test),   f"{team_test['target'].mean():.1%} victorias",
              len(player_test),  f"{player_test['target'].mean():.1f} pts prom",
              team_test.shape[1],  player_test.shape[1]],
}, index=['team filas','team target dist','player filas',
          'player target dist','team columnas','player columnas'])

## Guardado de datasets procesados

In [23]:
os.makedirs('../data/processed', exist_ok=True)

team_train.to_csv('../data/processed/team_classification_train.csv',   index=False)
team_test.to_csv('../data/processed/team_classification_test.csv',     index=False)
player_train.to_csv('../data/processed/player_regression_train.csv',  index=False)
player_test.to_csv('../data/processed/player_regression_test.csv',    index=False)

pd.DataFrame({
    'archivo': [
        'team_classification_train.csv',
        'team_classification_test.csv',
        'player_regression_train.csv',
        'player_regression_test.csv'
    ],
    'filas': [len(team_train), len(team_test), len(player_train), len(player_test)],
    'columnas': [team_train.shape[1], team_test.shape[1], player_train.shape[1], player_test.shape[1]]
})

,archivo,filas,columnas
0,team_classification_train.csv,1626,21
1,team_classification_test.csv,534,21
2,player_regression_train.csv,14404,19
3,player_regression_test.csv,4901,19


In [24]:
team_train.to_sql('ml_team_classification_train',   engine, if_exists='replace', index=False)
team_test.to_sql('ml_team_classification_test',     engine, if_exists='replace', index=False)
player_train.to_sql('ml_player_regression_train',  engine, if_exists='replace', index=False)
player_test.to_sql('ml_player_regression_test',    engine, if_exists='replace', index=False)

pd.DataFrame({
    'tabla PostgreSQL': [
        'ml_team_classification_train',
        'ml_team_classification_test',
        'ml_player_regression_train',
        'ml_player_regression_test'
    ],
    'estado': ['guardada'] * 4
})

,tabla PostgreSQL,estado
0,ml_team_classification_train,guardada
1,ml_team_classification_test,guardada
2,ml_player_regression_train,guardada
3,ml_player_regression_test,guardada


Los cuatro datasets se guardan tanto en `data/processed/` como en PostgreSQL. Los CSV son la fuente principal para el notebook de modelado y permiten cargar los datos sin depender de la base de datos. Las tablas en PostgreSQL garantizan la trazabilidad completa del pipeline.

## Resumen de decisiones técnicas

| Decisión tomada | Alternativa descartada | Justificación técnica |
|---|---|---|
| División temporal 75/25 | División aleatoria (`train_test_split`) | Los datos tienen dependencia temporal. La división aleatoria introduce data leakage: el modelo podría aprender de partidos de marzo para predecir partidos de octubre, algo imposible en producción real |
| Ventanas de 5 y 10 partidos | Estadísticas acumuladas de toda la temporada | Los promedios de temporada no capturan el estado de forma reciente. Un equipo con buen promedio anual pero en racha de 5 derrotas seguidas requiere ambas escalas para predecirse bien |
| `shift(1)` antes de cada `rolling` | Incluir el partido actual en el cálculo | Sin `shift(1)`, el feature del partido `t` contendría las estadísticas del propio partido `t`. Esto es data leakage directo: se estaría usando la respuesta para construir la pregunta |
| Filtro `min >= 5` antes del rolling | Conservar todos los registros | Apariciones inferiores a 5 minutos no representan el rendimiento real del jugador y sesgan los promedios móviles hacia abajo, introduciendo ruido en lugar de señal |
| `expanding().mean()` para `defense_rating` | Media fija de toda la temporada | La media fija usa datos futuros para caracterizar al rival en partidos tempranos de la temporada. La expanding mean replica el conocimiento real disponible antes de cada partido |
| `StandardScaler` aplicado en Pipeline 04 | `MinMaxScaler` aplicado al dataset | MinMaxScaler es sensible a outliers extremos. StandardScaler es más robusto. Aplicarlo en el Pipeline de sklearn garantiza que se ajusta solo sobre el train set en cada fold de validación cruzada |